# Phase 2 · 23020023
Chọn Kaggle CPU (Accelerator = None), bật Internet cho bước cài đặt, dùng một kernel mới. Chạy các ô theo thứ tự. Thiết lập biến môi trường PHASE2_B bằng SHA đầy đủ của main đã duyệt sau ba PR. Có thể chạy smoke và pilot trước khi có B. PHASE2_RUNNER là SHA tooling đã công bố.

In [ ]:
from pathlib import Path
import json, os, secrets, shutil, subprocess, sys, tempfile, uuid
from datetime import datetime, timezone

repo = "https://github.com/trandangduat/face-checkin-service.git"
base = "247db09d98f4cedcc9ad22fdbd112ba328324aa9"
runner_sha = os.environ["PHASE2_RUNNER"]
work = Path("/kaggle/working")
session = uuid.uuid4().hex
root = work / f"23020023-{session}"
root.mkdir()
private = Path(tempfile.mkdtemp(prefix="phase2-private-"))
os.environ.update(JWT_SECRET=secrets.token_urlsafe(32), SEED_ADMIN_PASSWORD=secrets.token_urlsafe(24),
                  SEED_USER_PASSWORD=secrets.token_urlsafe(24), ACCESS_TOKEN_EXPIRE_MINUTES="120",
                  EMBEDDING_MODEL="buffalo_s", SIMILARITY_THRESHOLD="0.40", CUDA_VISIBLE_DEVICES="",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", OMP_NUM_THREADS="1")
def call(argv, cwd=None, **kwargs):
    return subprocess.run(list(map(str, argv)), cwd=cwd, check=True, **kwargs)
if not shutil.which("uv"):
    call([sys.executable,"-m","pip","install","uv"])
def checkout(label, sha):
    path = private / label
    call(["git","clone",repo,path])
    call(["git","switch","--detach",sha],path)
    assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=path,text=True).strip() == sha
    call(["uv","sync","--python","3.12","--frozen","--extra","dev"],path)
    return path
runner = checkout("runner", runner_sha)
a = checkout("A", base)
def python(app):
    return app / ".venv/bin/python"
def bench(*args):
    call([python(runner),"-m","scripts.phase2_bench",*args],runner)
fx = private/"fixtures"
def cell(app, output, round_, scenario, gallery, c, *extra):
    bench("run","--app",app,"--python",python(app),"--fixtures",fx,"--output",output,
          "--round",round_,"--session",session,"--scenario",scenario,"--gallery",gallery,"--c",c,*extra)

In [ ]:
# Seed và cache model trước khi mở cửa sổ đo.
bench("prepare","--app",a,"--python",python(a),"--fixtures",fx)
shutil.copyfile(fx/"manifest.json", root/"fixture-manifest.json")
with (root/"baseline-smoke.log").open("x") as log:
    call([python(a),"-m","scripts.run_baseline","--backend","sqlite","--requests",6,
          "--history-requests",12,"--concurrency","1,2","--warmup",2,"--output-dir",root/"baseline-smoke"],
         a, stdout=log, stderr=subprocess.STDOUT)
for n in (1,2):
    cell(a,root/"smoke",n,"E1","pure",1,"--count",6)
    cell(a,root/"smoke",n,"E2","pure",2,"--count",12)
def matrix():
    for scenario, galleries, levels in [
        ("E1",["pure"],[1,2,4,8]), ("E2",["pure"],[1,2,4,8]),
        ("E3",["pure","mixed"],[1,4]), ("probe",["pure","mixed"],[1]), ("M1",["pure"],[1])]:
        for gallery in galleries:
            for c in levels:
                yield scenario,gallery,c
for settings in matrix():
    cell(a,root/"pilot",1,*settings)
freeze = dict(runner=runner_sha, A=base, rng=23020023, ttl_minutes=120,
              session=session, E1_gain=10, E3_mixed_probe_gain=20, E2_E3_pure_max_regression=10,
              E1_success_rps_max_loss=5, pilot_finished=datetime.now(timezone.utc).isoformat())
(root/"freeze.json").write_text(json.dumps(freeze,indent=2))

In [ ]:
def comparison(label, sha):
    candidate = checkout(label,sha)
    out = root/label
    out.mkdir()
    for app, app_sha in [(a,base),(candidate,sha)]:
        env = dict(os.environ,PHASE2_R1="1",PHASE2_R1_CSV=str(out/f"R1-23020023-{app_sha}.csv"))
        with (out/f"R1-23020023-{app_sha}.log").open("x") as log:
            tested = subprocess.run([python(app),"-c",
                "import sys,runpy;sys.path.append(sys.argv.pop(1));"
                "runpy.run_path(sys.argv.pop(1),run_name='__main__')",
                runner,runner/"tests/unit/test_phase2_bench.py","TransactionTests","-v"],
                cwd=app,env=env,stdout=log,stderr=subprocess.STDOUT)
        print("R1/parity",app_sha,tested.returncode)
    for n, order in enumerate(((a,candidate),(candidate,a),(a,candidate)),1):
        for app in order:
            for settings in matrix():
                try:
                    cell(app,out,n,*settings)
                except subprocess.CalledProcessError:
                    sc,gal,c = settings
                    fixture = {"E1":"single","E2":"history","M1":"single"}.get(sc,gal)
                    app_sha = base if app == a else sha
                    meta = out/app_sha/f"{sc}-{fixture}"/f"c{c}"/f"r{n}"/"metadata.json"
                    if not meta.exists() or json.loads(meta.read_text())["passed"]:
                        raise
                    print("Oracle failed",meta.relative_to(out))
    bench("compare","--output",out,"--session",session,"--a",base,"--b",sha,
          "--table",out/f"AB-23020023-{base[:7]}-{sha[:7]}.csv")
# Các nhánh đơn lẻ phải dùng đúng SHA cùng runner và snapshot.
for label,key in [("Son","PHASE2_SON"),("Thanh","PHASE2_THANH")]:
    if os.environ.get(key):
        comparison(label,os.environ[key])
comparison("integrated",os.environ["PHASE2_B"])